In [1]:
import ast
import pandas as pd
import glob
import json
import numpy as np
import tqdm
from langchain_text_splitters import MarkdownHeaderTextSplitter
import os

import sys
sys.path.append("../..")
from benchmark.src import create_sentence_nace_code_similarities

In [2]:
all_results_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/similarity_search_descriptions/"

In [3]:
df_overview = pd.read_csv("../../data/datasets/stoxx_600/stoxx_600_overview.csv", sep=";")
df_overview = df_overview.dropna(subset="Report")
#df_overview = df_overview.dropna(subset="description_page")

In [4]:
report_jsons = glob.glob("../../data/datasets/stoxx_600/JSONs/*.json")
report_jsons.sort()
report_jsons

['../../data/datasets/stoxx_600/JSONs/AAK AB1.json',
 '../../data/datasets/stoxx_600/JSONs/ABB Ltd.2.json',
 '../../data/datasets/stoxx_600/JSONs/ANDRITZ AG1.json',
 '../../data/datasets/stoxx_600/JSONs/ASM International N.V.1.json',
 '../../data/datasets/stoxx_600/JSONs/ASR Nederland N.V.1.json',
 '../../data/datasets/stoxx_600/JSONs/AXA SA1.json',
 '../../data/datasets/stoxx_600/JSONs/Aalberts N.V.1.json',
 '../../data/datasets/stoxx_600/JSONs/Accelleron Industries AG1.json',
 '../../data/datasets/stoxx_600/JSONs/Acciona SA2.json',
 '../../data/datasets/stoxx_600/JSONs/Accor SA1.json',
 '../../data/datasets/stoxx_600/JSONs/Ackermans & van Haaren NV1.json',
 '../../data/datasets/stoxx_600/JSONs/Adecco Group AG1.json',
 '../../data/datasets/stoxx_600/JSONs/Admiral Group plc1.json',
 '../../data/datasets/stoxx_600/JSONs/Airbus SE1.json',
 '../../data/datasets/stoxx_600/JSONs/Akzo Nobel N.V.2.json',
 '../../data/datasets/stoxx_600/JSONs/Alcon AG1.json',
 '../../data/datasets/stoxx_600/JS

In [58]:
with open(report_jsons[1], "r") as f: 
    text = json.load(f)

In [59]:
text

{'source_file': 'ABB Ltd.2.pdf',
 'num_pages': 236,
 'pages': [{'page': 1, 'markdown': ''},
  {'page': 2, 'markdown': 'HBB\n\n-\n\nIntegrated Report 2022\n\nDisha'},
  {'page': 3,
   'markdown': "-\n\nThe cover page shows our country and business offices in AI enabled, platinum rated green building Disha. It is located in our gold rated Peenya campus in Bengaluru. The presence of Disha building in the integrated Peenya campus symbolizes ABB India's collaborative way of working with a model of empowerment much like the first Integrated Annual Report of the Company. They are also testimony to ABB India's commitment to sustainability, deploying thousands of ABB products for resource and energy efficiency, creating safer and productive work environment with sustainable technology, and green energy for running operations."},
  {'page': 4,
   'markdown': "-\n\n## About ABB\n\nABB is a technology leader in electrification and automation, enabling a more sustainable and resource-efficient futu

### Try to find table of contents

In [26]:
from langchain_ollama import OllamaLLM

def query_ollama(prompt: str, model: str = "llama3.2:1b") -> str:
    """
    Simple function to call an Ollama model through LangChain.

    Args:
        prompt (str): The input text prompt for the model.
        model (str): The name of the Ollama model to use (default: "llama3").

    Returns:
        str: The model's generated response.
    """
    llm = OllamaLLM(model=model)
    response = llm.invoke(prompt)
    return response

In [ ]:
def get_table_of_contents(text: list): 
    

In [39]:
print(text["pages"][2]["markdown"])

## Contents

## This is AAK

| Making Better Happen ™                           | 3   |
|--------------------------------------------------|-----|
| A Scandinavian company with a global presence    | 4   |
| Our vision                                       | 5   |
| 2022 in brief                                    |     |
| Key events                                       | 6   |
| Key figures                                      | 7   |
| Message from the Chair and the CEO               | 8   |
| What we do                                       |     |
| AAK in the value chain                           | 10  |
| AAK - a Multi-oil Ingredient House               | 11  |
| A broad range of raw materials                   | 12  |
| A business model built on Making Better Happen ™ | 13  |
| Strategy and aspiration                          |     |
| Continuous development of our strategy           | 15  |
| Updated portfolio strategy                       | 16  |
| Strategic actions to real

In [60]:
filter_table_and_header = lambda x : "\n".join([line for line in  x.split("\n") if len(line) > 0 and (line[0] == "#" or line[0] == "|")])

pages = [filter_table_and_header(page["markdown"]) for page in text["pages"]]

pages

['',
 '',
 '',
 '## About ABB',
 '## Table of contents\n## 16 Value creation\n|   18 | Who we are                        |\n|------|-----------------------------------|\n|   20 | Divisions and Business areas      |\n|   22 | Building a deep footprint         |\n|   24 | Our purpose                       |\n|   28 | Our strategy and priorities       |\n|   30 | Our operating model - the ABB Way |\n## 34 Sustainability in practice',
 '## 88 Risk and opportunities\n## 104  Governance structure\n## 112  Statutory Report\n## 166  Financial Statements',
 '## Letter from the Chairman and the Managing Director\n## External environment\n## Megatrends and local impact\n## Strategy for consistent performance',
 '## Sustainability in practice\n## A stable foundation for future growth',
 '',
 '## Key figures at a glance and five-year summary\n|                                                   |        |        |        | ( ` in Crores)   | ( ` in Crores)   |\n|-------------------------------------

In [61]:
table_of_contents_prompt = f"You are an analyst of an annual report: Which of the following pages does contain the table of contents?\n {pages[:10]}\n Only return the page number."

In [62]:
result = query_ollama(table_of_contents_prompt)

In [63]:
result

'The table of contents is located on pages 16 and 34.'

In [68]:
pages

['',
 '',
 '',
 '## About ABB',
 '## Table of contents\n## 16 Value creation\n|   18 | Who we are                        |\n|------|-----------------------------------|\n|   20 | Divisions and Business areas      |\n|   22 | Building a deep footprint         |\n|   24 | Our purpose                       |\n|   28 | Our strategy and priorities       |\n|   30 | Our operating model - the ABB Way |\n## 34 Sustainability in practice',
 '## 88 Risk and opportunities\n## 104  Governance structure\n## 112  Statutory Report\n## 166  Financial Statements',
 '## Letter from the Chairman and the Managing Director\n## External environment\n## Megatrends and local impact\n## Strategy for consistent performance',
 '## Sustainability in practice\n## A stable foundation for future growth',
 '',
 '## Key figures at a glance and five-year summary\n|                                                   |        |        |        | ( ` in Crores)   | ( ` in Crores)   |\n|-------------------------------------

In [76]:
for page in pages[:10]: 
    if page != "": 
        
        is_toc_on_page_prompt = f"""You are a precise and detail-oriented financial analyst specialized in reading and interpreting corporate annual reports.

        Task:
        Determine whether the following page from an annual report contains a description of the company’s business segments — meaning explanations of what the company produces, sells, or earns revenue from.

        Input:
        {page}

        Output format:
        Respond only with one of the following words:
        - Yes — if the page includes a description of business segments, products, or revenue-generating activities.
        - No — if it does not.

        Additional notes:
        - Ignore sections like financial tables, risk factors, management introductions, or sustainability discussions unless they explicitly describe business activities or products.
        - Do not include any explanation or reasoning — output must be exactly one word: Yes or No."""
        

        answer = query_ollama(is_toc_on_page_prompt, "llama3:8b")
        print(answer)
        if "yes" in answer.strip().lower(): 
            print("Page :", page)
            break

No
No
No
No
No
No


In [73]:
full_pages = [page["markdown"] for page in text["pages"]]
for page in full_pages: 
    if page != "": 
        is_toc_on_page_prompt = f"You are an analyst of an annual report. Does the following page contain a description of the Business Segments of the company, i.e. what the company produces or makes money with?\n{page}\nOnly answer with yes or no."
        answer = query_ollama(is_toc_on_page_prompt, "llama3:8b")
        print(answer)
        print(page)
        if "yes" in answer.strip().lower(): 
            print(page)
            break

Yes
HBB

-

Integrated Report 2022

Disha
HBB

-

Integrated Report 2022

Disha


In [77]:
full_pages

['',
 'HBB\n\n-\n\nIntegrated Report 2022\n\nDisha',
 "-\n\nThe cover page shows our country and business offices in AI enabled, platinum rated green building Disha. It is located in our gold rated Peenya campus in Bengaluru. The presence of Disha building in the integrated Peenya campus symbolizes ABB India's collaborative way of working with a model of empowerment much like the first Integrated Annual Report of the Company. They are also testimony to ABB India's commitment to sustainability, deploying thousands of ABB products for resource and energy efficiency, creating safer and productive work environment with sustainable technology, and green energy for running operations.",
 "-\n\n## About ABB\n\nABB is a technology leader in electrification and automation, enabling a more sustainable and resource-efficient future.\n\nThe Company's solutions connect engineering know-how and software to optimize how things are manufactured, moved, powered and operated. Building on more than 130 y

In [75]:
is_toc_on_page_prompt

'You are an analyst of an annual report. Does the following page contain a description of the Business Segments of the company, i.e. what the company produces or makes money with?\nHBB\n\n-\n\nIntegrated Report 2022\n\nDisha\nOnly answer with yes or no.'

In [57]:
full_pages

['',
 'Annual Report 2022\n\n## The Multi-oil Ingredient House\n\n<!-- image -->',
 '## Contents\n\n## This is AAK\n\n| Making Better Happen ™                           | 3   |\n|--------------------------------------------------|-----|\n| A Scandinavian company with a global presence    | 4   |\n| Our vision                                       | 5   |\n| 2022 in brief                                    |     |\n| Key events                                       | 6   |\n| Key figures                                      | 7   |\n| Message from the Chair and the CEO               | 8   |\n| What we do                                       |     |\n| AAK in the value chain                           | 10  |\n| AAK - a Multi-oil Ingredient House               | 11  |\n| A broad range of raw materials                   | 12  |\n| A business model built on Making Better Happen ™ | 13  |\n| Strategy and aspiration                          |     |\n| Continuous development of our strategy  

In [74]:
full_pages

['',
 'HBB\n\n-\n\nIntegrated Report 2022\n\nDisha',
 "-\n\nThe cover page shows our country and business offices in AI enabled, platinum rated green building Disha. It is located in our gold rated Peenya campus in Bengaluru. The presence of Disha building in the integrated Peenya campus symbolizes ABB India's collaborative way of working with a model of empowerment much like the first Integrated Annual Report of the Company. They are also testimony to ABB India's commitment to sustainability, deploying thousands of ABB products for resource and energy efficiency, creating safer and productive work environment with sustainable technology, and green energy for running operations.",
 "-\n\n## About ABB\n\nABB is a technology leader in electrification and automation, enabling a more sustainable and resource-efficient future.\n\nThe Company's solutions connect engineering know-how and software to optimize how things are manufactured, moved, powered and operated. Building on more than 130 y